# Ordered Logistic Regression Results for Adoption Predictors: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR²](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

You'll use the Croissant schema URL and work entirely with entities referenced by their `@id`, following best practices for metadata-driven data science and reproducibility.

### Dataset Source

The dataset is described via a [Croissant schema](https://mlcommons.org/croissant/) URL. All explorations reference data entities by their Croissant `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load the dataset metadata and records using the Croissant schema and `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review the available record sets, along with their `@id`s and fields. All entities are referenced by their `@id` for reproducible data science.

In [ ]:
# List available record sets by @id and name
record_sets = dataset.record_sets()
print("Available Record Sets:")
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else 'Unnamed'}")

# For demonstration, list fields for each RecordSet with their @id
for rs in record_sets:
    print(f"\nFields for RecordSet @id={rs.id}:")
    fields = rs.fields
    for field in fields:
        print(f"    - @id: {field.id}, name: {getattr(field, 'name', 'UnnamedField')}")

## 3. Data Extraction

Load records from each record set (using its `@id`) as a pandas DataFrame. All field and record set references use the Croissant `@id`.

In [ ]:
# Extract all record sets into separate DataFrames (indexed by record_set @id)

dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} rows for RecordSet @id={record_set_id}")

# Display columns for each DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nColumns for RecordSet @id={record_set_id}:")
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's pick a record set of interest for EDA. We'll reference the RecordSet and all fields by their `@id`.

- We'll select a numeric field by its `@id` (shown in the previous overview). 
- We'll demonstrate filtering, normalizing, and grouping on the DataFrame using only `@id`s for all references.

> **Adjust the following variables** depending on your dataset: choose an actual record set and field `@id` for numeric and grouping analyses.

For demonstration, we'll auto-select the first record set and attempt numeric operations on its numeric columns if any.

In [ ]:
# Select a record set and numeric field
selected_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[selected_record_set_id]

# Attempt to auto-detect numeric fields by @id (column name)
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Selected numeric field @id: {numeric_field_id}")
else:
    print("No numeric field found in the record set.")
    numeric_field_id = None

# Filter: Values > threshold
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].count() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nRecords with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    # Normalization
    df_copy = filtered_df.copy()
    df_copy[f"{numeric_field_id}_normalized"] = (df_copy[numeric_field_id] - df_copy[numeric_field_id].mean()) / df_copy[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(df_copy[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by a categorical field (try to select a non-numeric field @id)
    group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field_id = group_candidates[0] if group_candidates else None

    if group_field_id:
        grouped_df = df_copy.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id} (all by @id):")
        display(grouped_df.head())
    else:
        print("No groupable categorical field found in the record set.")
else:
    print("Skipping EDA; no numeric fields found.")

## 5. Visualization

Visualize the relationship between a numeric field (referenced by its `@id`) and a grouping or categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution if numeric field exists
if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If grouping field exists, show boxplot
    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No numeric field detected; skipping visualization.')

## 6. Conclusion

In this notebook, you:
- Loaded a FAIR² dataset from its Croissant schema using the `mlcroissant` Python library.
- Explored available record sets and fields via their `@id` values for reproducibility and clarity.
- Extracted and processed data using only entity `@id` references as required by best Croissant practices.
- Performed basic exploratory analysis and visualization, grouped and normalized by `@id` fields.

You can adapt the code to target specific record sets or fields by updating the `@id` values as needed.